# Dostrajanie modelu e5-przemkow v2 na parach Qwena

Notatnik startuje od już dostrojonego modelu `e5-przemkow` i trenuje go na nowych parach z `pary_qwen_czyste.jsonl`.

**Wgraj:** `pary_qwen_czyste.jsonl`, `fragmenty.jsonl`, `e5-przemkow.zip`

**WAŻNE:** menu *Środowisko wykonawcze → Zmień typ środowiska → T4 GPU*.

In [ ]:
# 1. Instalacja bibliotek
!pip install -q sentence-transformers==3.4.1 datasets accelerate

In [ ]:
# 2. Wgranie plików
import json
from google.colab import files, drive

# Zamontuj Google Drive (model e5-przemkow.zip pobieramy stąd)
print("Montowanie Google Drive...")
drive.mount('/content/drive')

# Podaj ścieżkę do e5-przemkow.zip na Twoim Drive:
SCIEZKA_ZIP = "/content/drive/MyDrive/e5-przemkow.zip"
print(f"Kopiuję model z Drive: {SCIEZKA_ZIP}")
import shutil, os
shutil.copy(SCIEZKA_ZIP, "e5-przemkow.zip")
print("ZIP skopiowany.")

# Wgraj dwa małe pliki normalnie przez upload
print("Wgraj pary_qwen_czyste.jsonl i fragmenty.jsonl:")
uploaded = files.upload()

# Wczytaj pary treningowe
pary = []
with open("pary_qwen_czyste.jsonl", encoding="utf-8") as f:
    for linia in f:
        pary.append(json.loads(linia))
print(f"Wczytano {len(pary)} par pytanie-fragment.")

# Wczytaj pełny korpus do ewaluacji
fragmenty = []
with open("fragmenty.jsonl", encoding="utf-8") as f:
    for linia in f:
        fragmenty.append(json.loads(linia))
print(f"Wczytano {len(fragmenty)} fragmentów korpusu.")


In [ ]:
# 3. Zbiór treningowy i testowy
import random
from datasets import Dataset

random.seed(42)
random.shuffle(pary)
n_test = max(50, int(0.1 * len(pary)))
test, train = pary[:n_test], pary[n_test:]
print(f"Trening: {len(train)} | Test: {n_test}")

def format_passage(p):
    return f"passage: {p['tytul']}. {p['tekst']}"

train_ds = Dataset.from_dict({
    "anchor":   ["query: " + p["pytanie"] for p in train],
    "positive": [format_passage(p) for p in train],
})
print("Zbiór treningowy gotowy.")

In [ ]:
# 4. Trening kontrastywny (MultipleNegativesRankingLoss)
from sentence_transformers import (SentenceTransformer,
                                   SentenceTransformerTrainer,
                                   SentenceTransformerTrainingArguments)
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers.training_args import BatchSamplers

# Startujemy od już dostrojonego modelu, nie od bazowego!
!unzip -q e5-przemkow.zip
BAZOWY = "e5-przemkow"
model = SentenceTransformer(BAZOWY)
loss = MultipleNegativesRankingLoss(model)

args = SentenceTransformerTrainingArguments(
    output_dir="trening",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    fp16=True,
    logging_steps=10,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    save_strategy="no",
    report_to=[],
)

trainer = SentenceTransformerTrainer(
    model=model, args=args, train_dataset=train_ds, loss=loss)
trainer.train()

model.save_pretrained("e5-przemkow")
print("Model zapisany w katalogu e5-przemkow/")

In [ ]:
# 5. Ewaluacja: Recall@5 przed i po dostrojeniu
from sentence_transformers import SentenceTransformer, util

korpus_teksty = [format_passage(d) for d in fragmenty]
korpus_id = [d["id"] for d in fragmenty]

def recall_at_5(m):
    emb_k = m.encode(korpus_teksty, batch_size=128, convert_to_tensor=True,
                     normalize_embeddings=True, show_progress_bar=True)
    emb_q = m.encode(["query: " + p["pytanie"] for p in test],
                     batch_size=128, convert_to_tensor=True,
                     normalize_embeddings=True)
    trafienia = util.semantic_search(emb_q, emb_k, top_k=5)
    ok = 0
    for i, p in enumerate(test):
        top5 = [korpus_id[h["corpus_id"]] for h in trafienia[i]]
        if p["id"] in top5:
            ok += 1
    return ok / len(test)

bazowy = SentenceTransformer(BAZOWY)  # e5-przemkow jako punkt startowy
r_przed = recall_at_5(bazowy)
print(f"Recall@5 PRZED (e5-przemkow v1): {r_przed:.1%}")

dostrojony = SentenceTransformer("e5-przemkow")
r_po = recall_at_5(dostrojony)
print(f"Recall@5 PO dostrojeniu (e5-przemkow v2): {r_po:.1%}")
print(f"Poprawa: +{(r_po - r_przed):.1%}")

In [ ]:
# 6. Spakowanie i pobranie modelu
!zip -r -q e5-przemkow.zip e5-przemkow
from google.colab import files
files.download("e5-przemkow.zip")

## Wdrożenie na serwerze

1. Pobierz `e5-przemkow.zip` z Colaba
2. Wgraj do kontenera 101 (podmień stary model)
3. Usuń `chroma_db` i przeindeksuj (~6h)
